# Parametrizing a ligand or a modified residue for AMBER

This notebook follows the class tutorial step by step. Each step **explains what it is about to do**, **shows the exact command**, runs it, and **checks the result**. At the end, `commands.sh` contains every command in order, so you can run them yourself later.

**How to use it:** run the cells from top to bottom with ▶ (or Shift+Enter). Read each step's output before running the next one.

| Step | Tutorial | Ligand | Modified residue |
|---|---|---|---|
| Look at the structure | Part 2 | ✓ | ✓ |
| Add ACE / NME caps | Part 2.3 | | ✓ |
| Charges and atom types | Part 3 | ✓ | ✓ |
| Backbone atom types | Part 4 | | ✓ |
| Library (.lib / .prepi) | Part 5 / 6 | ✓ | ✓ |
| Missing parameters | Part 7 | ✓ | ✓ |
| Validation | Part 8 | ✓ | ✓ |

## 0. Install AmberTools and solerita  *(about 1–2 minutes, once per session)*

AmberTools is installed with **micromamba**, a small, fast version of conda, into its own folder. Colab's own Python is left untouched, so there is no need to restart the session.

In [ ]:
#@title Install (run once) { display-mode: "form" }
#@markdown Leave **solerita_source** empty and you will be asked to upload `solerita.zip`. Or paste a GitHub address, e.g. `https://github.com/USER/solerita`.
solerita_source = ""  #@param {type:"string"}

import os, subprocess, sys, time, shutil
t0 = time.time()
AMBER = "/content/amber"
def sh(cmd):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if r.returncode != 0:
        print(r.stdout[-2000:], r.stderr[-2000:]); raise RuntimeError(cmd)
    return r.stdout

if not os.path.exists(f"{AMBER}/bin/antechamber"):
    print("Downloading micromamba ...")
    # save to a file first: piping straight into tar breaks (SIGPIPE) once tar has its file
    sh("curl -Ls https://micro.mamba.pm/api/micromamba/linux-64/latest -o /content/mm.tar.bz2 && "
       "tar -xjf /content/mm.tar.bz2 -C /content bin/micromamba")
    print("Installing AmberTools from conda-forge (the slow part) ...")
    sh(f"MAMBA_ROOT_PREFIX=/content/mamba /content/bin/micromamba create -y -q "
       f"-p {AMBER} -c conda-forge ambertools")
os.environ["PATH"] = f"{AMBER}/bin:" + os.environ["PATH"]
os.environ["AMBERHOME"] = AMBER

if solerita_source.strip():
    url = solerita_source.strip()
    sh(f"pip install -q 'git+{url}'" if "github.com" in url else f"pip install -q '{url}'")
else:
    try:
        import solerita  # already installed this session
    except ImportError:
        from google.colab import files
        print("Upload solerita.zip:")
        up = files.upload()
        name = next(iter(up))
        sh(f"rm -rf /content/solerita_src && mkdir -p /content/solerita_src && "
           f"unzip -q -o '{name}' -d /content/solerita_src")
        root = [d for d, _, f in os.walk("/content/solerita_src") if "pyproject.toml" in f][0]
        sh(f"pip install -q '{root}'")
import importlib, site; importlib.invalidate_caches()
sys.path[:0] = [p for p in site.getsitepackages() if p not in sys.path]
import solerita
print(sh(f"{sys.executable} -m solerita --check"))
print(f"Ready in {time.time()-t0:.0f} s.")

## 1. Choose your molecule

Type a three-letter code from the **PDB Chemical Component Dictionary (CCD)**. You can browse it at <https://www.rcsb.org/> → search the code → *Ligand* page. The tutorial uses:

* `CER`: ceramide, a **ligand** (a free molecule)
* `SME`: methionine sulfoxide, a **modified residue** (part of a protein chain)

**The net charge is yours to state.** It decides which chemical species the charges describe, and nothing in the program can know it for you.

If the PDB cannot be reached, tick *offline*: CER and SME are shipped with solerita.

In [ ]:
#@title Molecule { display-mode: "form" }
code = "SME"        #@param {type:"string"}
net_charge = 0      #@param {type:"integer"}
offline = False     #@param {type:"boolean"}

from solerita import Parametrization
os.chdir("/content")
p = Parametrization(code.strip().upper(), charge=net_charge, offline=offline)

## 2. Look at the starting structure  — *Part 2*

Read the summary carefully: number of atoms, hydrogens, formal charge, stereocentres. **Is this the molecule, and the protonation state, you meant?** Everything after this step describes exactly this structure.

In [ ]:
p.inspect()

## 3. Caps  — *Part 2.3, modified residues only*

Skip this cell for a ligand: it will tell you so.

In [ ]:
if p.kind == "residue":
    p.cap()
else:
    print("A ligand is not capped. Go on to the next step.")

### Have a look

The capped residue (or the ligand) in 3D. Rotate it with the mouse.

In [ ]:
#@title 3D view { display-mode: "form" }
try:
    import py3Dmol
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "py3Dmol"]); import py3Dmol
v = py3Dmol.view(width=600, height=400)
v.addModel(open(p.path(f"{p.resname}.pdb")).read(), "pdb")
v.setStyle({"stick": {}}); v.addPropertyLabels("atom", {"elem": "H", "invert": True},
    {"fontSize": 10, "showBackground": False, "fontColor": "black"})
v.zoomTo(); v.show()

## 4. Charges and atom types  — *Part 3*

The slowest step: `sqm` runs a semi-empirical quantum calculation. From a few seconds up to a few minutes.

In [ ]:
p.charges()

## 5. Backbone atom types  — *Part 4, modified residues only*

In [ ]:
if p.kind == "residue":
    p.backbone()
else:
    print("Ligands keep their GAFF2 types. Go on to the next step.")

## 6. The library and the missing parameters  — *Parts 5, 6 and 7*

The order differs between the branches, and the cell takes care of it:

* **ligand:** missing parameters first (LEaP needs them to build the `.lib`), then the library
* **residue:** the `.prepi` first (`parmchk2` reads it), then the missing parameters

In [ ]:
if p.kind == "ligand":
    p.parameters()
    p.library()
else:
    p.library()
    p.parameters()

## 7. Validation  — *Part 8*

LEaP must build the molecule (for a residue, inside the peptide ALA‑X‑ALA), agree on its charge, and a short minimisation must lower the energy.

In [ ]:
p.test()

## 8. Summary, and your files

`commands.sh` is the whole procedure as a script: open it and compare it with the tutorial.

In [ ]:
p.finish()
print()
print(open(p.path("commands.sh")).read())

In [ ]:
#@title Download everything as a zip { display-mode: "form" }
import shutil
z = shutil.make_archive(f"/content/{p.resname}_parameters", "zip", p.workdir)
try:
    from google.colab import files; files.download(z)
except ImportError:
    print("Saved", z)

---
### When something fails

Each check that fails says **what to do**. The most common causes:

* **the net charge** does not match the structure. Look again at the hydrogens in step 2.
* **sqm does not converge**: atoms too close together in the starting structure.
* **LEaP cannot find a parameter**: read the line it prints; it names the atom types.

After fixing the cause, re-run the cell of the step that failed. Steps after it have to be run again too.

### Want to try another molecule?

Go back to cell 1, change the code and the charge, and run everything from there. Some to try: `CFF` (caffeine, a ligand), `MLY` (dimethyl-lysine) or `SEP` (phosphoserine).

For the two modified residues, **count the hydrogens in step 2 before choosing the charge.** The CCD entry is one particular protonation state, and it may not be the one found at pH 7. If you want a different one, edit the structure yourself and give solerita the file.